# Dynamic and Seasonal Surface-Water Mapping and Change Detection over India Using NISAR L- and S-Band SAR


In [ ]:
import os
import glob
import h5py
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.transform import from_origin
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from scipy.ndimage import distance_transform_edt, label

# Base paths
DATA_DIR = r"C:\indu\ingarss\data"
OUTPUT_DIR = r"C:\indu\ingarss\outputs"
PLOTS_DIR = os.path.join(OUTPUT_DIR, "plots")
os.makedirs(PLOTS_DIR, exist_ok=True)

def get_h5_path(band, date):
    pattern = os.path.join(DATA_DIR, f"{band}_band", date, "*.h5")
    matches = glob.glob(pattern)
    return matches[0] if matches else None

l_jul_path = get_h5_path("L", "2026-07-14")
s_jul_path = get_h5_path("S", "2026-07-14")
l_oct_path = get_h5_path("L", "2026-10-18")
s_oct_path = get_h5_path("S", "2026-10-18")
l_aug_path = get_h5_path("L", "2026-08-21")
s_aug_path = get_h5_path("S", "2026-08-21")

print(f"L-band July 14:  {os.path.basename(l_jul_path)}")
print(f"S-band July 14:  {os.path.basename(s_jul_path)}")
print(f"L-band Oct 18:   {os.path.basename(l_oct_path)}")
print(f"S-band Oct 18:   {os.path.basename(s_oct_path)}")
print(f"L-band Aug 21:   {os.path.basename(l_aug_path)} (Quad-Pol)")
print(f"S-band Aug 21:   {os.path.basename(s_aug_path)} (Compact-Pol)")


### 1. Data Ingestion, Geodetic Co-Registration & Radiometric Calibration


In [ ]:
def load_and_coregister(l_path, s_path):
    # 1. Master L-band (GCOV frequencyB, 80m spacing)
    with h5py.File(l_path, "r") as fl:
        gl = fl["science/LSAR/GCOV/grids/frequencyB"]
        epsg = int(gl["projection"][()])
        dx_l = float(gl["xCoordinateSpacing"][()])
        dy_l = float(gl["yCoordinateSpacing"][()])
        xs_l = gl["xCoordinates"][:]
        ys_l = gl["yCoordinates"][:]
        hh_l = gl["HHHH"][:].astype(np.float32)
        hv_l = gl["HVHV"][:].astype(np.float32)
        mask_l = gl["mask"][:]
        
    l_valid = (mask_l > 0) & (mask_l != 255) & (hh_l > 0) & (hv_l > 0)
    l_hh_db = np.full_like(hh_l, np.nan)
    l_hv_db = np.full_like(hv_l, np.nan)
    l_hh_db[l_valid] = 10.0 * np.log10(hh_l[l_valid])
    l_hv_db[l_valid] = 10.0 * np.log10(hv_l[l_valid])
    
    # 2. S-band at stride 8 (10m * 8 = 80m effective grid)
    with h5py.File(s_path, "r") as fs:
        gs = fs["science/SSAR/GCOV/grids/frequencyA"]
        dx_s = float(gs["xCoordinateSpacing"][()])
        dy_s = float(gs["yCoordinateSpacing"][()])
        xs_s = gs["xCoordinates"][:]
        ys_s = gs["yCoordinates"][:]
        s = 8
        hh_s = gs["HHHH"][::s, ::s].astype(np.float32)
        hv_s = gs["HVHV"][::s, ::s].astype(np.float32)
        mask_s = gs["mask"][::s, ::s]
        
    dx_s_eff = dx_s * s
    dy_s_eff = dy_s * s
    xs_s_eff = xs_s[::s]
    ys_s_eff = ys_s[::s]
    
    s_valid_raw = (mask_s > 0) & (mask_s != 255) & (hh_s > 0) & (hv_s > 0)
    hh_s_lin = np.full_like(hh_s, np.nan)
    hv_s_lin = np.full_like(hv_s, np.nan)
    hh_s_lin[s_valid_raw] = hh_s[s_valid_raw]
    hv_s_lin[s_valid_raw] = hv_s[s_valid_raw]
    
    # Precise outer geotransform definitions
    src_transform = from_origin(xs_s_eff[0] - dx_s_eff/2.0, ys_s_eff[0] - dy_s_eff/2.0, dx_s_eff, abs(dy_s_eff))
    dst_transform = from_origin(xs_l[0] - dx_l/2.0, ys_l[0] - dy_l/2.0, dx_l, abs(dy_l))
    
    s_hh_lin_aligned = np.full_like(hh_l, np.nan)
    s_hv_lin_aligned = np.full_like(hv_l, np.nan)
    
    # Reproject in LINEAR power domain with explicit NaN NoData
    reproject(source=hh_s_lin, destination=s_hh_lin_aligned, src_transform=src_transform,
              src_crs=rasterio.crs.CRS.from_epsg(epsg), dst_transform=dst_transform,
              dst_crs=rasterio.crs.CRS.from_epsg(epsg), resampling=Resampling.bilinear,
              src_nodata=np.nan, dst_nodata=np.nan)
    reproject(source=hv_s_lin, destination=s_hv_lin_aligned, src_transform=src_transform,
              src_crs=rasterio.crs.CRS.from_epsg(epsg), dst_transform=dst_transform,
              dst_crs=rasterio.crs.CRS.from_epsg(epsg), resampling=Resampling.bilinear,
              src_nodata=np.nan, dst_nodata=np.nan)
              
    s_valid = ~np.isnan(s_hh_lin_aligned) & ~np.isnan(s_hv_lin_aligned) & (s_hh_lin_aligned > 0) & (s_hv_lin_aligned > 0)
    s_hh_db = np.full_like(s_hh_lin_aligned, np.nan)
    s_hv_db = np.full_like(s_hv_lin_aligned, np.nan)
    s_hh_db[s_valid] = 10.0 * np.log10(s_hh_lin_aligned[s_valid])
    s_hv_db[s_valid] = 10.0 * np.log10(s_hv_lin_aligned[s_valid])
    
    common_valid = l_valid & s_valid
    return (hh_l, hv_l, l_hh_db, l_hv_db, l_valid,
            s_hh_lin_aligned, s_hv_lin_aligned, s_hh_db, s_hv_db, s_valid,
            common_valid, dx_l, dy_l, xs_l, ys_l, epsg, dst_transform)

print("Loading and co-registering July 14, 2026 NISAR granules...")
(l_lin_jul, l_hv_lin_jul, l_hh_jul, l_hv_jul, l_val_jul,
 s_lin_jul, s_hv_lin_jul, s_hh_jul, s_hv_jul, s_val_jul,
 val_jul, dx_l, dy_l, xs_l, ys_l, epsg, dst_transform) = load_and_coregister(l_jul_path, s_jul_path)

print("Loading and co-registering October 18, 2026 NISAR granules...")
(l_lin_oct, l_hv_lin_oct, l_hh_oct, l_hv_oct, l_val_oct,
 s_lin_oct, s_hv_lin_oct, s_hh_oct, s_hv_oct, s_val_oct,
 val_oct, _, _, _, _, _, _) = load_and_coregister(l_oct_path, s_oct_path)

pixel_km2 = (abs(dx_l) * abs(dy_l)) / 1e6
print(f"\n[SUCCESS] Real NISAR Data Ingested! Grid: {l_hh_jul.shape[0]} x {l_hh_jul.shape[1]} pixels.")
print(f"Pixel Resolution: {abs(dx_l):.1f}m x {abs(dy_l):.1f}m | Pixel Area: {pixel_km2:.6f} km².")
print(f"Valid Coverage: July = {np.sum(val_jul)*pixel_km2:,.2f} km² | Oct = {np.sum(val_oct)*pixel_km2:,.2f} km².")


### 2. Dynamic Single-Frequency Water Classification & Polarimetric Sandbar De-Aliasing

In [5]:
def adaptive_otsu(vals, search_range=(-24.0, -11.5)):
    counts, edges = np.histogram(vals, bins=256, range=(-35.0, 5.0))
    centers = (edges[:-1] + edges[1:]) / 2.0
    total = np.sum(counts)
    if total == 0: return -16.0
    p = counts / total
    w = np.cumsum(p)
    mu = np.cumsum(p * centers)
    sb = (mu[-1] * w - mu)**2 / (w * (1.0 - w) + 1e-10)
    in_range = (centers >= search_range[0]) & (centers <= search_range[1])
    return float(centers[in_range][np.argmax(sb[in_range])])

# Compute independent dynamic thresholds per sensor and season
th_l_jul = adaptive_otsu(l_hh_jul[val_jul], (-24.0, -11.5))
th_s_jul = adaptive_otsu(s_hh_jul[val_jul], (-24.0, -11.5))
th_l_oct = adaptive_otsu(l_hh_oct[val_oct], (-24.0, -12.0))
th_s_oct = adaptive_otsu(s_hh_oct[val_oct], (-24.0, -12.0))

print(f"Dynamic Optimal Thresholds: July L = {th_l_jul:.2f} dB, S = {th_s_jul:.2f} dB | Oct L = {th_l_oct:.2f} dB, S = {th_s_oct:.2f} dB")

# Polarimetric NDPI filter 
ndpi_jul = (l_lin_jul - l_hv_lin_jul) / (l_lin_jul + l_hv_lin_jul + 1e-6)
ndpi_oct = (l_lin_oct - l_hv_lin_oct) / (l_lin_oct + l_hv_lin_oct + 1e-6)

# Raw candidate detections
raw_w_l_jul = (l_hh_jul < th_l_jul) & (l_hv_jul < -21.0) & l_val_jul
raw_w_s_jul = (s_hh_jul < th_s_jul) & (s_hv_jul < -20.0) & s_val_jul

raw_w_l_oct = (l_hh_oct < th_l_oct) & (l_hv_oct < -21.0) & l_val_oct
raw_w_s_oct = (s_hh_oct < th_s_oct) & (s_hv_oct < -20.0) & s_val_oct

# Clean water masks filtered by NDPI 
w_l_jul = raw_w_l_jul & (ndpi_jul > 0.65)
w_s_jul = raw_w_s_jul
w_l_oct = raw_w_l_oct & (ndpi_oct > 0.65)
w_s_oct = raw_w_s_oct

sandbars_jul = raw_w_l_jul & ~w_l_jul
suppression_rate_jul = (np.sum(sandbars_jul) / max(1, np.sum(raw_w_l_jul))) * 100

print(f"\n--- Polarimetric Sandbar Suppression (July 14) ---")
print(f"Raw L-band Water:       {np.sum(raw_w_l_jul):,} pixels ({np.sum(raw_w_l_jul)*pixel_km2:,.2f} km²)")
print(f"Clean L-band Water:     {np.sum(w_l_jul):,} pixels ({np.sum(w_l_jul)*pixel_km2:,.2f} km²)")
print(f"Suppressed Sandbars:    {np.sum(sandbars_jul):,} pixels ({np.sum(sandbars_jul)*pixel_km2:,.2f} km², {suppression_rate_jul:.1f}% eliminated)")


### 3. Physically Rigorous Dual-Frequency (L+S) Fusion via Linear NDFI

In [ ]:
def fuse_water_linear(lin_l, lin_s, db_l_hh, db_l_hv, db_s_hh, db_s_hv, w_l, w_s, valid):
    denom = lin_l + lin_s
    ndfi = np.full_like(lin_l, 0.0)
    v_denom = valid & (denom > 1e-6)
    ndfi[v_denom] = (lin_l[v_denom] - lin_s[v_denom]) / denom[v_denom]
    
    # Asymmetric fusion components
    w_core = w_l & w_s
    w_l_rec = w_l & (db_l_hv < -21.0) & ((ndfi < -0.05) | (db_s_hh < -10.5)) & valid
    w_s_rec = w_s & (db_s_hv < -20.0) & (db_l_hh < -10.0) & valid
    
    w_fused = (w_core | w_l_rec | w_s_rec) & valid
    return w_fused, ndfi

fused_jul, ndfi_jul = fuse_water_linear(l_lin_jul, s_lin_jul, l_hh_jul, l_hv_jul, s_hh_jul, s_hv_jul, w_l_jul, w_s_jul, val_jul)
fused_oct, ndfi_oct = fuse_water_linear(l_lin_oct, s_lin_oct, l_hh_oct, l_hv_oct, s_hh_oct, s_hv_oct, w_l_oct, w_s_oct, val_oct)

# Quantitative Metrics
inter_jul = np.sum(w_l_jul & w_s_jul)
union_jul = np.sum(w_l_jul | w_s_jul)
iou_jul = (inter_jul / union_jul) * 100 if union_jul > 0 else 0
dice_jul = (2.0 * inter_jul / (np.sum(w_l_jul) + np.sum(w_s_jul))) * 100 if (np.sum(w_l_jul) + np.sum(w_s_jul)) > 0 else 0
gain_over_l = (np.sum(fused_jul) - np.sum(w_l_jul)) / max(1, np.sum(w_l_jul)) * 100
gain_over_s = (np.sum(fused_jul) - np.sum(w_s_jul)) / max(1, np.sum(w_s_jul)) * 100

print(f"--- FUSION EVALUATION (July 2026) ---")
print(f"L vs S Spatial IoU:      {iou_jul:.2f}% (Dice: {dice_jul:.2f}%)")
print(f"Fused Extent (July 14):  {np.sum(fused_jul):,} pixels ({np.sum(fused_jul)*pixel_km2:,.2f} km²)")
print(f"Fused Extent (Oct 18):   {np.sum(fused_oct):,} pixels ({np.sum(fused_oct)*pixel_km2:,.2f} km²)")
print(f"Added Water over L-only: +{gain_over_l:.2f}% (+{(np.sum(fused_jul) - np.sum(w_l_jul))*pixel_km2:,.2f} km²)")
print(f"Added Water over S-only: +{gain_over_s:.2f}% (+{(np.sum(fused_jul) - np.sum(w_s_jul))*pixel_km2:,.2f} km²)")


### 4. Multi-Temporal Dynamics & Hydrological Spatial Partitioning

In [ ]:
common_val = val_jul & val_oct
change_map = np.full_like(fused_jul, 255, dtype=np.uint8)  # 255 = NoData

change_map[common_val & (fused_jul == 0) & (fused_oct == 0)] = 0  # Stable Land
change_map[common_val & (fused_jul == 1) & (fused_oct == 1)] = 1  # Persistent Water
change_map[common_val & (fused_jul == 1) & (fused_oct == 0)] = 2  # Recession / Drying
change_map[common_val & (fused_jul == 0) & (fused_oct == 1)] = 3  # Newly Detected Water

pers_pix = np.sum(change_map == 1)
reced_pix = np.sum(change_map == 2)
new_pix = np.sum(change_map == 3)
land_pix = np.sum(change_map == 0)

print(f"--- Multi-Temporal Dynamics (July Peak vs. October Recession) ---")
print(f"Persistent Water:     {pers_pix:,} pixels ({pers_pix*pixel_km2:,.2f} km²)")
print(f"Floodwater Recession: {reced_pix:,} pixels ({reced_pix*pixel_km2:,.2f} km²)")
print(f"Newly Detected Water: {new_pix:,} pixels ({new_pix*pixel_km2:,.2f} km²)")
print(f"Stable Non-Water:     {land_pix:,} pixels ({land_pix*pixel_km2:,.2f} km²)")

# Hydrological Partitioning using Major River Channels
persistent_mask = (change_map == 1)
labeled_pers, num_features = label(persistent_mask)
sizes = np.bincount(labeled_pers.ravel())
sizes[0] = 0  # Ignore background

# Major continuous river network (> 500 connected pixels = > 3.2 km²)
river_labels = np.where(sizes >= 500)[0]
river_channel_mask = np.isin(labeled_pers, river_labels) if len(river_labels) > 0 else persistent_mask

dist_to_river_m = distance_transform_edt(~river_channel_mask) * abs(dx_l)
new_water_mask = (change_map == 3)

near_channel_water = new_water_mask & (dist_to_river_m <= 1500.0)
floodplain_chaur_water = new_water_mask & (dist_to_river_m > 1500.0)

print(f"\n--- Hydrological Partitioning of Newly Detected Water ---")
print(f"Near-Channel Lateral Expansion (<=1.5km): {np.sum(near_channel_water):,} px ({np.sum(near_channel_water)*pixel_km2:,.2f} km², {np.sum(near_channel_water)/max(1, new_pix)*100:.1f}%)")
print(f"Inland Floodplain Depressions (>1.5km):    {np.sum(floodplain_chaur_water):,} px ({np.sum(floodplain_chaur_water)*pixel_km2:,.2f} km², {np.sum(floodplain_chaur_water)/max(1, new_pix)*100:.1f}%)")


### 6. Results Visualization & Publication Exports

In [ ]:
ext = [xs_l[0] - dx_l/2.0, xs_l[-1] + dx_l/2.0, ys_l[-1] - abs(dy_l)/2.0, ys_l[0] + abs(dy_l)/2.0]
cmap_w = ListedColormap(['#ffffff', '#08519c'])

# Figure 1: Dual-Frequency Fusion
fig, axes = plt.subplots(1, 3, figsize=(20, 6.5), dpi=200)

axes[0].imshow(w_l_jul, cmap=ListedColormap(['#ffffff', '#1f78b4']), vmin=0, vmax=1, extent=ext, origin='upper')
axes[0].set_title(f'A. Clean L-Band Only Water (July 14)\nArea: {np.sum(w_l_jul)*pixel_km2:,.1f} km² (NDPI De-Aliased)', fontsize=11, fontweight='bold')
axes[0].set_xlabel('UTM Easting (m)'); axes[0].set_ylabel('UTM Northing (m)')
axes[0].grid(True, linestyle=':', alpha=0.3)

axes[1].imshow(w_s_jul, cmap=ListedColormap(['#ffffff', '#33a02c']), vmin=0, vmax=1, extent=ext, origin='upper')
axes[1].set_title(f'B. S-Band Only Water (July 14)\nArea: {np.sum(w_s_jul)*pixel_km2:,.1f} km² (Misses canopy water)', fontsize=11, fontweight='bold')
axes[1].set_xlabel('UTM Easting (m)'); axes[1].set_ylabel('UTM Northing (m)')
axes[1].grid(True, linestyle=':', alpha=0.3)

axes[2].imshow(fused_jul, cmap=cmap_w, vmin=0, vmax=1, extent=ext, origin='upper')
axes[2].set_title(f'C. Combined Fused Water (L+S)\nArea: {np.sum(fused_jul)*pixel_km2:,.1f} km² (+{gain_over_l:.1f}% over L)', fontsize=11, fontweight='bold')
axes[2].set_xlabel('UTM Easting (m)'); axes[2].set_ylabel('UTM Northing (m)')
axes[2].grid(True, linestyle=':', alpha=0.3)

plt.suptitle('Figure 1: Dual-Frequency (L+S) Water Extraction (July 14, 2026 Monsoon Peak)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
p1 = os.path.join(PLOTS_DIR, 'Dual_Frequency_Fusion.png')
plt.savefig(p1, dpi=200, bbox_inches='tight')
plt.show()

# Figure 2: Multi-Temporal Dynamics
fig, axes = plt.subplots(1, 3, figsize=(22, 7), dpi=200)
cmap_dyn = ListedColormap(['#f7f7f7', '#08519c', '#e6550d', '#2ca02c'])

im0 = axes[0].imshow(fused_jul, cmap=cmap_w, vmin=0, vmax=1, extent=ext, origin='upper')
axes[0].set_title(f'A. July 14, 2026 (Monsoon Peak)\nTotal Water: {np.sum(fused_jul)*pixel_km2:,.1f} km²', fontsize=11, fontweight='bold')
axes[0].set_xlabel('UTM Easting (m)'); axes[0].set_ylabel('UTM Northing (m)')
axes[0].grid(True, linestyle=':', alpha=0.3)
cb0 = plt.colorbar(im0, ax=axes[0], ticks=[0.25, 0.75], fraction=0.046, pad=0.04)
cb0.ax.set_yticklabels(['Dry Land', 'Water'], fontsize=9)

im1 = axes[1].imshow(fused_oct, cmap=cmap_w, vmin=0, vmax=1, extent=ext, origin='upper')
axes[1].set_title(f'B. October 18, 2026 (Post-Monsoon)\nTotal Water: {np.sum(fused_oct)*pixel_km2:,.1f} km²', fontsize=11, fontweight='bold')
axes[1].set_xlabel('UTM Easting (m)'); axes[1].set_ylabel('UTM Northing (m)')
axes[1].grid(True, linestyle=':', alpha=0.3)
cb1 = plt.colorbar(im1, ax=axes[1], ticks=[0.25, 0.75], fraction=0.046, pad=0.04)
cb1.ax.set_yticklabels(['Dry Land', 'Water'], fontsize=9)

im2 = axes[2].imshow(change_map, cmap=cmap_dyn, vmin=0, vmax=3, extent=ext, origin='upper')
axes[2].set_title(f'C. Multi-Temporal Dynamics\nPersistent: {pers_pix*pixel_km2:,.0f} km² | Receded: {reced_pix*pixel_km2:,.0f} km² | New: {new_pix*pixel_km2:,.0f} km²', fontsize=11, fontweight='bold')
axes[2].set_xlabel('UTM Easting (m)'); axes[2].set_ylabel('UTM Northing (m)')
axes[2].grid(True, linestyle=':', alpha=0.3)
cb2 = plt.colorbar(im2, ax=axes[2], ticks=[0.375, 1.125, 1.875, 2.625], fraction=0.046, pad=0.04)
cb2.ax.set_yticklabels(['Stable Land', 'Persistent', 'Recession', 'Newly Detected'], fontsize=9)

plt.suptitle('Figure 2: Multi-Temporal Inundation Dynamics (July 14 Peak vs. October 18 Recession)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
p2 = os.path.join(PLOTS_DIR, 'Multi_Temporal_Dynamics.png')
plt.savefig(p2, dpi=200, bbox_inches='tight')
plt.show()

# Figure 3: Hydrological Distance Partitioning & Polarimetric Validation
part_map = np.zeros_like(change_map, dtype=np.uint8)
part_map[near_channel_water] = 1
part_map[floodplain_chaur_water] = 2

sand_map = np.zeros_like(change_map, dtype=np.uint8)
sand_map[w_l_jul] = 1
sand_map[sandbars_jul] = 2

fig, axes = plt.subplots(1, 2, figsize=(16, 7), dpi=200)

im_e = axes[0].imshow(part_map, cmap=ListedColormap(['#f7f7f7', '#005a32', '#74c476']), vmin=0, vmax=2, extent=ext, origin='upper')
axes[0].set_title(f'A. Hydrological Distance Partitioning (New Water)\nNear-Channel (<=1.5km): {np.sum(near_channel_water)*pixel_km2:,.1f} km² ({np.sum(near_channel_water)/max(1, new_pix)*100:.1f}%) | Inland Chaurs: {np.sum(floodplain_chaur_water)*pixel_km2:,.1f} km²', fontsize=11, fontweight='bold')
axes[0].set_xlabel('UTM Easting (m)'); axes[0].set_ylabel('UTM Northing (m)')
axes[0].grid(True, linestyle=':', alpha=0.3)
cb_e = plt.colorbar(im_e, ax=axes[0], ticks=[0.33, 1.0, 1.67], fraction=0.046, pad=0.04)
cb_e.ax.set_yticklabels(['Other Land', 'Near-Channel (<=1.5km)', 'Inland Chaurs (>1.5km)'], fontsize=9)

im_h = axes[1].imshow(sand_map, cmap=ListedColormap(['#f7f7f7', '#08519c', '#e41a1c']), vmin=0, vmax=2, extent=ext, origin='upper')
axes[1].set_title(f'B. Polarimetric Sandbar De-Aliasing (July 14)\nClean Water (Blue) | Suppressed Sandbars (Red, {suppression_rate_jul:.1f}% eliminated)', fontsize=11, fontweight='bold')
axes[1].set_xlabel('UTM Easting (m)'); axes[1].set_ylabel('UTM Northing (m)')
axes[1].grid(True, linestyle=':', alpha=0.3)
cb_h = plt.colorbar(im_h, ax=axes[1], ticks=[0.33, 1.0, 1.67], fraction=0.046, pad=0.04)
cb_h.ax.set_yticklabels(['Dry Land', 'Clean Water', 'Suppressed Sandbars'], fontsize=9)

plt.suptitle('Figure 3: Hydrological Distance Partitioning & Polarimetric Validation', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
p3 = os.path.join(PLOTS_DIR, 'Hydrological_and_Polarimetric_Validation.png')
plt.savefig(p3, dpi=200, bbox_inches='tight')
plt.show()
